In [ ]:
from dataloading import *
from bpetokenizer import *

label_vocab, cl_train, cl_dev, cl_test, it_train, it_dev, it_test = load_rr()
id_to_label = {v: k for k, v in label_vocab.items()}
tokenizer = load_bpe_tokenizer()

In [ ]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from model_arch import *
class Phase3ClassificationModel(nn.Module):
    def __init__(self, base_model, num_classes=13, pad_token_id=0):
        super().__init__()
        self.base_model = base_model
        self.num_classes = num_classes
        self.pad_token_id = pad_token_id
        
        self.classification_head = nn.Linear(self.base_model.cfg.d_model, num_classes)

    def forward(self, idx):
        B, T = idx.shape
        
        x = self.base_model.tok_emb(idx) + self.base_model.pos_emb[:T]
        for block in self.base_model.blocks:
            x = block(x)
        hidden_states = self.base_model.ln_f(x)
        
        mask = (idx != self.pad_token_id).float().unsqueeze(-1)
        sum_embeddings = (hidden_states * mask).sum(dim=1)
        real_token_counts = mask.sum(dim=1)
        
        pooled_output = sum_embeddings / torch.clamp(real_token_counts, min=1e-9)
        
        logits = self.classification_head(pooled_output)
        
        return logits

class LoRMALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 8, alpha: float = 8.0):
        super().__init__()
        self.base = base_layer
        self.scale = alpha / r
        
        for param in self.base.parameters():
            param.requires_grad = False
            
        d = base_layer.out_features 
        
        self.lorma_A = nn.Linear(d, r, bias=False)
        self.lorma_B = nn.Linear(r, d, bias=False)
        
        nn.init.zeros_(self.lorma_B.weight)

    def forward(self, x):
        z = self.base(x)
        return z + self.scale * self.lorma_B(self.lorma_A(z))

In [ ]:
cfg = LMConfig(vocab_size=50260)
base_lm = DecoderLM(cfg)
model = Phase3ClassificationModel(base_lm, num_classes=13, pad_token_id=0)

rank = 8
alpha_val = 8.0

for block in model.base_model.blocks:
    block.attn.q_proj = LoRMALinear(block.attn.q_proj, r=rank, alpha=alpha_val)
    block.attn.v_proj = LoRMALinear(block.attn.v_proj, r=rank, alpha=alpha_val)

model.load_state_dict(torch.load("checkpoint.pt", map_location=device, weights_only=True))
model.to(device)
model.eval()

In [ ]:
import json
predictions = {}

def predict_document(doc_sentences):
    doc_preds = []
    for sentence in doc_sentences:
        encoded = tokenizer.encode(sentence)[:1024]
        input_ids = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)
        
        with torch.no_grad():
            logits = model(input_ids)
            pred_id = torch.argmax(logits, dim=1).item()
            
        doc_preds.append(id_to_label[pred_id])
    return doc_preds

for doc_id in cl_test:
    doc_sentences = cl_test[doc_id]
    predictions[doc_id] = predict_document(doc_sentences)

for doc_id in it_test:
    doc_sentences = it_test[doc_id]
    predictions[doc_id] = predict_document(doc_sentences)

with open("prediction.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=4)